In [6]:
from ragas import evaluate
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langsmith import traceable
from google import genai
from deepeval.test_case import LLMTestCase
from deepeval import evaluate
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric
)
from Backend.Main_pipeline import retrieval,rerank,genereate
from pydantic import BaseModel
from typing import Optional
from pathlib import Path
import time
import random
import json
load_dotenv()

ModuleNotFoundError: No module named 'Backend'

In [ ]:
try:
    base_path = Path(__file__).parent
    with open(base_path.parent / 'json_files' /'final_data.json', 'r', encoding='utf-8') as file:
        data = json.load(file)
except FileNotFoundError:
    print("Error: The file final_data.json does not exist.")  

In [11]:
data[0]

{'text': 'To generate text from a large language model we’ll just generalize this model a bit: at each step we’ll sample tokens according to their probability conditioned on our previous choices, and we’ll use the large language model as the probability model that tells us this probability.\n\n12 CHAPTER 7 • LARGE LANGUAGE MODELS\n\nrandom\n\nsaan\n\nsampling\n\nThe algorithm is called random sampling, or random multinomial sampling (because we are sampling from a multinomial distribution across words). We can formalize random sampling as follows: we are generating a sequence of tokens {w1,w2,...,wN} until we hit the end-of-sequence token, using x ∼ p(x) to mean ‘choose x by sampling from the distribution p(x)’:\n\ni←1\n\nwi ∼ p(w) while wi != EOS i←i + 1 wi ∼ p(wi | w<i)\n\nu\n\ny\n\nsoftmax —probabilities— sample a word\n\nTransformer (or other decoder)\n\nthe\n\nSo.\n\nlong\n\nand thanks for\n\n?\n\nFigure 7.9 Random multinomial sampling: we randomly chose a word according to its pr

In [ ]:

client = genai.Client()  


class ChunkAssessment(BaseModel):
    has_enough_context: bool
    reason: str
    question: Optional[str] = None
    ground_truth: Optional[str] = None

def generate_qa(chunk, max_retries=3):
    prompt = f"""Assess if this text chunk has enough self-contained context 
to generate a meaningful factual question without needing surrounding context.

If yes → generate a question and answer ONLY based on information 
present in this chunk. The question must be answerable from this chunk alone,
without needing any other text or context.
Do NOT generate questions that require comparing or combining 
information from multiple sources.
If no → set question and ground_truth to null.

Text: {chunk['text']}"""

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-3.1-flash-lite",
                contents=prompt,
                config={
                    "response_mime_type": "application/json",
                    "response_schema": ChunkAssessment,
                }
            )
            
            result = response.parsed 
            
            if not result.has_enough_context:
                print(f"⏭️ Skipping: {result.reason}")
                return None
            
            return {
                "question": result.question,
                "ground_truth": result.ground_truth,
                "source": chunk['metadata']["filename"],
                "page": chunk['metadata']["pages"]
            }
            
        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # exponential backoff: 1s, 2s, 4s
                print(f"⚠️ Attempt {attempt+1} failed: {e}. Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"❌ Failed after {max_retries} attempts: {e}")
                return None

# main loop
dataset = []
target_questions=25
random.shuffle(data)

for i, chunk in enumerate(data):
    if len(dataset) >= target_questions:
        print(f"✅ Reached target of {target_questions} questions")
        break
    print(f"Processing {i+1}/{len(data)}...")
    
    qa = generate_qa(chunk)
    if qa:
        dataset.append(qa)
        print(f"✅ {qa['question'][:60]}")
    
   
    time.sleep(4)

print(f"\nGenerated {len(dataset)} questions")

In [ ]:
@traceable(name="RAG Evaluation Pipeline")
def eval_pipeline(item):
    chunks = retrieval(item["question"], k_val=20)
    reranked = rerank(chunks, item["question"], final_k=5)
    answer = genereate(reranked, item["question"])
    
    return {
        "question": item["question"],
        "answer": answer,
        "contexts": [c[0].page_content for c in reranked],
        "ground_truth": item["ground_truth"],
        "source":item['source'],
        "page_number":item['page']
    }

eval_dataset = []

for item in dataset:
    print(f"Running: {item['question'][:60]}")
    result = eval_pipeline(item)
    eval_dataset.append(result)
    time.sleep(4)

In [15]:

dense_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5", model_kwargs={'device':'mps'})


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7127.75it/s]


In [ ]:

test_cases = [
    LLMTestCase(
        input=item["question"],
        actual_output=item["answer"],
        retrieval_context=item["contexts"],
        expected_output=item["ground_truth"]
    )
    for item in eval_dataset
]

In [38]:
metrics = [
    FaithfulnessMetric(threshold=0.7, model='gpt-4.1-mini'),
    AnswerRelevancyMetric(threshold=0.7, model='gpt-4.1-mini'),
    ContextualPrecisionMetric(threshold=0.7, model='gpt-4.1-mini'),
    ContextualRecallMetric(threshold=0.7, model='gpt-4.1-mini')
]

In [40]:
result=evaluate(test_cases=test_cases,metrics=metrics)

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              In causal self-attention, what range of inputs does the model have access to when    │
│  │                         processing a specific item?                                                          │
│  │     Actual Output:      In causal (or backward-looking) self-attention, when processing a specific item      │
│  │                         in an input sequence, the model has access to all input elements up to and           │
│  │                         including the current one, but it does not have access to information about any      │
│  │                         inputs beyond the current one.                                                       │
│  │                                                                                                              │
│  │                         This detail can be found on **page 4** and **page 5** of the file                    │
│  │                         **Transformers.pdf**.                                                                │
│  │     Expected Output:    The model has access to all inputs up to and including the one under                 │
│  │                         consideration, but no access to information about inputs beyond the current one.     │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Faithfulness         │ 1.00  │ 0.70      │ The score is 1.00 because there are no contradi...    │
│        PASS  │ Answer Relevancy     │ 1.00  │ 0.70      │ The score is 1.00 because the response directly...    │
│        PASS  │ Contextual Precision │ 0.83  │ 0.70      │ The score is 0.83 because the top-ranked nodes ...    │
│        FAIL  │ Contextual Recall    │ 0.50  │ 0.70      │ The score is 0.50 because the single sentence in      │
│              │                      │       │           │ the expected output is directly supported by the      │
│              │                      │       │           │ first node in the retrieval context, which            │
│              │                      │       │           │ explicitly states the model's access to inputs up     │
│              │                      │       │           │ to the current one only. However, since there is      │
│              │                      │       │           │ only one sentence and no additional supporting        │
│              │                      │       │           │ context, the recall is moderate rather than           │
│              │                      │       │           │ complete.                                             │
│                                                          

⚠ WARNING: No hyperparameters logged.
» ]8;id=390981;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 114.77s | token cost: 0.14270040000000003 USD)
» Test Results (25 total tests):
   » Pass Rate: 68.0% | Passed: 17 | Failed: 8

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [41]:
result

EvaluationResult(test_results=[TestResult(name='test_case_10', success=True, metrics_data=[MetricData(name='Faithfulness', threshold=0.7, success=True, score=1.0, reason='The score is 1.00 because there are no contradictions between the actual output and the retrieval context, indicating complete alignment and faithfulness.', strict_mode=False, evaluation_model='gpt-4.1-mini', error=None, evaluation_cost=0.0019456000000000002, input_tokens=2848, output_tokens=504, verbose_logs='Truths (limit=None):\n[\n    "Sampling methods have parameters that trade off quality and diversity in text generation.",\n    "Methods emphasizing the most probable words produce generations rated as more accurate, coherent, and factual but also more boring and repetitive.",\n    "Methods giving more weight to middle-probability words tend to be more creative and diverse but less factual and more likely incoherent or low-quality.",\n    "Perplexity is a way to evaluate language models by measuring how well they

In [42]:
import pandas as pd

In [44]:
df = pd.DataFrame(result)
df.to_csv("deepeval_results.csv", index=False)

In [45]:
result.model_dump()

{'test_results': [{'name': 'test_case_10',
   'success': True,
   'metrics_data': [{'name': 'Faithfulness',
     'threshold': 0.7,
     'success': True,
     'score': 1.0,
     'reason': 'The score is 1.00 because there are no contradictions between the actual output and the retrieval context, indicating complete alignment and faithfulness.',
     'strict_mode': False,
     'evaluation_model': 'gpt-4.1-mini',
     'error': None,
     'evaluation_cost': 0.0019456000000000002,
     'input_tokens': 2848,
     'output_tokens': 504,
     'verbose_logs': 'Truths (limit=None):\n[\n    "Sampling methods have parameters that trade off quality and diversity in text generation.",\n    "Methods emphasizing the most probable words produce generations rated as more accurate, coherent, and factual but also more boring and repetitive.",\n    "Methods giving more weight to middle-probability words tend to be more creative and diverse but less factual and more likely incoherent or low-quality.",\n    "P

In [ ]:
with open(base_path.parent / 'json_files' /"deepeval_results.json", "w") as f:
    json.dump(result.model_dump(), f, indent=2)


df = pd.DataFrame(result)
df.to_csv("deepeval_results.csv", index=False)